# Base simulada do imobilizado – Suzano S.A. (31/12/2025)
**Dados SINTÉTICOS** – trabalho acadêmico da disciplina Auditoria 1 (UnB, 2026.2). Não representam registros reais da Suzano.

Preparador: matrícula 241003769. Saldos de referência: nota 15 da DFP 2025 (consolidado).

**Como usar:** rode os blocos em ordem, de cima para baixo (Shift + Enter em cada um). Ao final, baixe os arquivos no último bloco.

## Bloco 1 – Configuração
Define a **semente** (para a base sair sempre igual), os **saldos da nota 15** que a base precisa respeitar, as subclasses de bens com a vida útil de cada uma, as unidades da Suzano e os usuários fictícios do sistema.

In [ ]:
# =====================================================================
# BLOCO 1 – Configuração
# Base SINTÉTICA do imobilizado da Suzano S.A. – trabalho acadêmico (UnB)
# Preparador: matrícula 241003769
# =====================================================================
import numpy as np
import pandas as pd

SEMENTE = 42                       # garante que a base sai sempre igual
rng = np.random.default_rng(SEMENTE)
DATA_BASE = pd.Timestamp('2025-12-31')

# Saldos da nota 15 (consolidado, 31/12/2025) em R$ mil -> convertidos para centavos
NOTA15 = {  # classe: (custo, depreciação acumulada, quantidade de bens)
    'Terrenos':                             (16_720_186,          0,   8_000),
    'Imóveis':                              (14_589_194,  4_659_310,  25_000),
    'Máquinas, equipamentos e instalações': (66_353_229, 32_174_476, 299_850),
    'Imobilizado em andamento':             ( 2_764_799,          0,  12_000),
    'Outros':                               ( 1_830_011,  1_127_446, 175_000),
}
CENT = 100_000  # R$ mil -> centavos (1.000 reais x 100 centavos)

# Subclasses, vida útil (anos) e peso de cada uma dentro da classe
SUBCLASSES = {
    'Terrenos': [('Fazenda florestal', None, 0.6), ('Terreno industrial', None, 0.4)],
    'Imóveis': [('Edifício industrial', 33, 0.5), ('Edifício administrativo', 30, 0.2),
                ('Armazém', 28, 0.2), ('Estrada florestal', 25, 0.1)],
    'Máquinas, equipamentos e instalações': [
        ('Caldeira de recuperação', 20, 0.05), ('Digestor', 20, 0.05), ('Máquina de papel', 18, 0.10),
        ('Secadora de celulose', 15, 0.10), ('Turbina e gerador', 18, 0.05), ('Equipamento florestal', 8, 0.25),
        ('Instalação elétrica', 12, 0.15), ('Bomba e motor', 10, 0.25)],
    'Imobilizado em andamento': [('Projeto de expansão', None, 0.3), ('Projeto de manutenção', None, 0.7)],
    'Outros': [('Veículo', 5, 0.3), ('Móveis e utensílios', 10, 0.3), ('Equipamento de informática', 5, 0.4)],
}

UNIDADES = [  # (unidade, UF, peso)
    ('Aracruz', 'ES', .15), ('Mucuri', 'BA', .12), ('Imperatriz', 'MA', .10), ('Três Lagoas', 'MS', .14),
    ('Ribas do Rio Pardo', 'MS', .18), ('Limeira', 'SP', .06), ('Jacareí', 'SP', .07), ('Mogi das Cruzes', 'SP', .03),
    ('Belém', 'PA', .02), ('Eunápolis', 'BA', .04), ('Maracanaú', 'CE', .02), ('Pine Bluff', 'EUA', .07)]
UNID_GARANTIA = {'Ribas do Rio Pardo', 'Três Lagoas', 'Imperatriz'}   # nota 15.1
USUARIOS = [f'USR{i:03d}' for i in range(1, 26)]
APROVADORES = [f'APR{i:02d}' for i in range(1, 9)]
print('Configuração carregada. Total de bens:', sum(v[2] for v in NOTA15.values()) + 150)

## Bloco 2 – Geração dos bens
Cria os 519.850 bens "originais", classe por classe. Os valores seguem uma distribuição **lognormal**: a maioria dos bens é pequena e poucos são muito grandes, como acontece num imobilizado real. As datas de aquisição vão de 1995 a 2025; os bens de Ribas do Rio Pardo (fábrica nova) ficam entre 2022 e 2024.

In [ ]:
# =====================================================================
# BLOCO 2 – Geração dos bens, classe por classe
# =====================================================================
def dia_util(datas):
    """Empurra datas de sábado/domingo para a segunda-feira seguinte."""
    dow = datas.dt.dayofweek
    return datas + pd.to_timedelta(np.where(dow == 5, 2, np.where(dow == 6, 1, 0)), unit='D')

partes = []
for classe, (custo_mil, dep_mil, n) in NOTA15.items():
    nomes, vidas, pesos = zip(*SUBCLASSES[classe])
    sub_idx = rng.choice(len(nomes), size=n, p=np.array(pesos) / sum(pesos))
    unid_idx = rng.choice(len(UNIDADES), size=n, p=[u[2] for u in UNIDADES])

    # Valores: distribuição lognormal (muitos bens pequenos, poucos muito grandes)
    sigma = {'Terrenos': 1.6, 'Imóveis': 1.8, 'Máquinas, equipamentos e instalações': 2.0,
             'Imobilizado em andamento': 1.9, 'Outros': 1.2}[classe]
    custo = rng.lognormal(mean=0, sigma=sigma, size=n)

    # Datas de aquisição: obras recentes; Ribas do Rio Pardo entre 2022 e 2024; demais 1995–2025
    if classe == 'Imobilizado em andamento':
        ini, fim = pd.Timestamp('2023-01-01'), DATA_BASE
        dias = rng.integers(0, (fim - ini).days, size=n)
    else:
        ini = pd.Timestamp('1995-01-01')
        dias = (rng.beta(2.2, 1.3, size=n) * (DATA_BASE - ini).days).astype(int)
    aquis = pd.Series(ini + pd.to_timedelta(dias, unit='D'))
    ribas = np.array([UNIDADES[i][0] == 'Ribas do Rio Pardo' for i in unid_idx])
    if classe != 'Imobilizado em andamento':
        r_ini = pd.Timestamp('2022-01-01')
        aquis[ribas] = r_ini + pd.to_timedelta(rng.integers(0, 1095, size=ribas.sum()), unit='D')
    aquis = dia_util(aquis)

    vida = np.array([vidas[i] if vidas[i] else np.nan for i in sub_idx], dtype=float)
    vida = np.where(np.isnan(vida), np.nan, vida + rng.integers(-2, 3, size=n))   # pequena variação por bem

    partes.append(pd.DataFrame({
        'classe': classe,
        'subclasse': [nomes[i] for i in sub_idx],
        'unidade': [UNIDADES[i][0] for i in unid_idx],
        'uf': [UNIDADES[i][1] for i in unid_idx],
        'data_aquisicao': aquis.values,
        'vida_util_anos': vida,
        'custo_bruto': custo,
    }))
    print(f'{classe}: {n:,} bens gerados')

base = pd.concat(partes, ignore_index=True)

## Bloco 3 – Ajuste à nota 15
Escala custo e depreciação acumulada para que a soma de cada classe fique **igual, em centavos,** ao saldo da nota 15. A depreciação de cada bem é proporcional ao tempo de uso dividido pela vida útil.

In [ ]:
# =====================================================================
# BLOCO 3 – Custo e depreciação ajustados aos saldos da nota 15 (em centavos)
# =====================================================================
def ajusta_total(valores, alvo_centavos):
    """Escala os valores para somarem exatamente o alvo (resíduo vai para o maior item)."""
    v = np.floor(valores / valores.sum() * alvo_centavos).astype(np.int64)
    v[np.argmax(v)] += alvo_centavos - v.sum()
    return v

base['custo_aquisicao'] = 0
base['depreciacao_acumulada'] = 0
anos_uso = ((DATA_BASE - base['data_aquisicao']).dt.days / 365.25).clip(lower=0)
for classe, (custo_mil, dep_mil, _) in NOTA15.items():
    m = base['classe'] == classe
    base.loc[m, 'custo_aquisicao'] = ajusta_total(base.loc[m, 'custo_bruto'].values, custo_mil * CENT)
    if dep_mil:
        fracao = (anos_uso[m] / base.loc[m, 'vida_util_anos']).clip(upper=1).values
        bruto = base.loc[m, 'custo_aquisicao'].values * fracao
        dep = ajusta_total(bruto, dep_mil * CENT)
        base.loc[m, 'depreciacao_acumulada'] = np.minimum(dep, base.loc[m, 'custo_aquisicao'].values)

base = base.drop(columns='custo_bruto')
base['custo_aquisicao'] = base['custo_aquisicao'].astype(np.int64)
base['depreciacao_acumulada'] = base['depreciacao_acumulada'].astype(np.int64)
print('Custo e depreciação ajustados à nota 15.')

## Bloco 4 – Demais colunas
Cria identificação (id, plaqueta, descrição), centro de custo, fornecedor, nota fiscal e os dados do lançamento (data, hora, usuário e aprovador), usados no teste de lançamentos do PT 240.

In [ ]:
# =====================================================================
# BLOCO 4 – Demais colunas (identificação, lançamento, situação)
# =====================================================================
n = len(base)
base.insert(0, 'id_bem', [f'SIM-{i:07d}' for i in range(1, n + 1)])
base.insert(1, 'plaqueta', rng.permutation(np.arange(100_000, 100_000 + n)))
base['descricao'] = base['subclasse'] + ' – ' + base['unidade'] + ' – item ' + (base.index + 1).astype(str)
base['centro_custo'] = base['uf'] + '-' + pd.Series(rng.integers(100, 999, size=n)).astype(str)
base['taxa_depreciacao_aa'] = (1 / base['vida_util_anos']).round(4)
base['data_inicio_depreciacao'] = (base['data_aquisicao'] + pd.Timedelta(days=30)).where(base['vida_util_anos'].notna())
base['fornecedor_id'] = 'FOR' + pd.Series(rng.integers(1, 3_000, size=n)).astype(str).str.zfill(4)
base['numero_nf'] = pd.Series(rng.integers(1, 999_999, size=n)).astype(str).str.zfill(6) + '-' + base.index.astype(str)
base['data_lancamento'] = dia_util(base['data_aquisicao'] + pd.to_timedelta(rng.integers(0, 10, size=n), unit='D'))
base['hora_lancamento'] = pd.Series(rng.integers(8, 18, size=n)).astype(str).str.zfill(2) + ':' + \
                          pd.Series(rng.integers(0, 60, size=n)).astype(str).str.zfill(2)
base['usuario_lancamento'] = rng.choice(USUARIOS, size=n)
base['usuario_aprovador'] = rng.choice(APROVADORES, size=n)
base['em_garantia'] = np.where(base['unidade'].isin(UNID_GARANTIA) & (base['classe'] != 'Outros'), 'Sim', 'Não')
base['situacao'] = np.where(base['classe'] == 'Imobilizado em andamento', 'Em construção', 'Em operação')
print('Colunas de identificação e lançamento criadas.')

## Bloco 5 – Erros inseridos (gabarito)
Insere os 10 tipos de erro (E1 a E10) em 1.600 bens e grava cada um no **gabarito**, com a afirmação afetada e o valor da distorção. O E10 infla o custo de 400 bens em R$ 150 milhões no total – acima da distorção tolerável de R$ 125,3 milhões (PT 320).

In [ ]:
# =====================================================================
# BLOCO 5 – Inserção dos erros (E1 a E10) e registro no gabarito
# =====================================================================
gabarito = []
def registra(ids, codigo, descricao, afirmacao, distorcao_centavos=0):
    for i, d in zip(ids, np.broadcast_to(distorcao_centavos, len(ids))):
        gabarito.append({'id_bem': i, 'codigo_erro': codigo, 'descricao_erro': descricao,
                         'afirmacao': afirmacao, 'distorcao_reais': round(int(d) / 100, 2)})

usados = set()
def sorteia(mascara, k):
    """Escolhe k bens ainda sem erro dentro da máscara."""
    idx = np.flatnonzero(mascara & ~base.index.isin(list(usados)))
    escolhidos = rng.choice(idx, size=k, replace=False)
    usados.update(escolhidos)
    return escolhidos

deprec = base['classe'].isin(['Imóveis', 'Máquinas, equipamentos e instalações', 'Outros'])

# E10 – custo inflado (superavaliação total próxima de R$ 150 milhões)
alvo_e10 = 150_000_000 * 100
cand = deprec & base['custo_aquisicao'].between(50_000_000, 5_000_000_000)   # bens de R$ 500 mil a R$ 50 mi
ix = sorteia(cand, 400)
taint = rng.uniform(0.05, 0.60, size=400)                        # % do valor contábil que está inflado
infl = base.loc[ix, 'custo_aquisicao'].values * taint / (1 - taint)
infl = np.round(infl / infl.sum() * alvo_e10).astype(np.int64)
base.loc[ix, 'custo_aquisicao'] += infl
registra(base.loc[ix, 'id_bem'], 'E10', 'Custo registrado acima do documento suporte', 'Existência / valorização', infl)
custo_correto = pd.Series(base['custo_aquisicao'].values, index=base.index)
custo_correto.loc[ix] -= infl

# E2 – depreciação acumulada maior que o custo
ix = sorteia(deprec, 100)
base.loc[ix, 'depreciacao_acumulada'] = (base.loc[ix, 'custo_aquisicao'] * rng.uniform(1.05, 1.30, 100)).astype(np.int64)
registra(base.loc[ix, 'id_bem'], 'E2', 'Depreciação acumulada maior que o custo', 'Valorização')

# E3 – terreno depreciado
ix = sorteia(base['classe'] == 'Terrenos', 80)
base.loc[ix, 'depreciacao_acumulada'] = (base.loc[ix, 'custo_aquisicao'] * rng.uniform(.05, .30, 80)).astype(np.int64)
registra(base.loc[ix, 'id_bem'], 'E3', 'Terreno com depreciação', 'Valorização', base.loc[ix, 'depreciacao_acumulada'].values * -1)

# E4 – obra em andamento depreciada
ix = sorteia(base['classe'] == 'Imobilizado em andamento', 80)
base.loc[ix, 'depreciacao_acumulada'] = (base.loc[ix, 'custo_aquisicao'] * rng.uniform(.05, .20, 80)).astype(np.int64)
registra(base.loc[ix, 'id_bem'], 'E4', 'Imobilizado em andamento com depreciação', 'Valorização', base.loc[ix, 'depreciacao_acumulada'].values * -1)

# E5 – data de aquisição posterior à data-base
ix = sorteia(deprec & (base['custo_aquisicao'] < 100_000_000), 120)
base.loc[ix, 'data_aquisicao'] = pd.Timestamp('2026-01-05') + pd.to_timedelta(rng.integers(0, 40, 120), unit='D')
registra(base.loc[ix, 'id_bem'], 'E5', 'Data de aquisição posterior a 31/12/2025', 'Corte / existência',
         base.loc[ix, 'custo_aquisicao'].values - base.loc[ix, 'depreciacao_acumulada'].values)

# E7 – vida útil fora da política
ix = sorteia(deprec, 120)
base.loc[ix, 'vida_util_anos'] = rng.choice([0, 99], size=120)
registra(base.loc[ix, 'id_bem'], 'E7', 'Vida útil fora da política (0 ou 99 anos)', 'Valorização')

# E8 – lançamento em fim de semana, aprovado pelo próprio usuário
ix = sorteia(base.index >= 0, 200)
base.loc[ix, 'data_lancamento'] = pd.Timestamp('2025-03-01') + pd.to_timedelta(rng.integers(0, 40, 200) * 7, unit='D')  # sábados
base.loc[ix, 'usuario_aprovador'] = base.loc[ix, 'usuario_lancamento']
registra(base.loc[ix, 'id_bem'], 'E8', 'Lançamento em fim de semana sem segregação (aprovador = usuário)', 'Ocorrência (fraude)')

# E9 – valor redondo alto lançado em 31/12 à noite por usuário incomum
ix = sorteia(deprec, 150)
base.loc[ix, 'custo_aquisicao'] = rng.choice([50_000, 100_000, 200_000], size=150) * 100
base.loc[ix, 'data_lancamento'] = DATA_BASE
base.loc[ix, 'hora_lancamento'] = '23:' + pd.Series(rng.integers(10, 59, 150)).astype(str).values
base.loc[ix, 'usuario_lancamento'] = 'USR099'
base.loc[ix, 'depreciacao_acumulada'] = 0
base.loc[ix, 'data_aquisicao'] = DATA_BASE
registra(base.loc[ix, 'id_bem'], 'E9', 'Valor redondo lançado em 31/12 às 23h por usuário incomum, sem documento', 'Ocorrência (fraude)',
         base.loc[ix, 'custo_aquisicao'].values - base.loc[ix, 'depreciacao_acumulada'].values)

# E1 – bens duplicados (mesma nota fiscal, fornecedor e valor)
ix = sorteia(deprec & (base['custo_aquisicao'] < 20_000_000), 150)
dup = base.loc[ix].copy()
dup['id_bem'] = [f'SIM-{i:07d}' for i in range(n + 1, n + 151)]
dup['plaqueta'] = np.arange(900_000, 900_150)
base = pd.concat([base, dup], ignore_index=True)
custo_correto = pd.concat([custo_correto, pd.Series(0, index=range(n, n + 150))])
registra(dup['id_bem'], 'E1', 'Bem duplicado (mesma NF, fornecedor e valor)', 'Existência',
         (dup['custo_aquisicao'] - dup['depreciacao_acumulada']).values)

# Valor contábil e E6 – valor contábil diferente de custo – depreciação
base['valor_contabil'] = base['custo_aquisicao'] - base['depreciacao_acumulada']
ix = sorteia(deprec.reindex(base.index, fill_value=False), 200)
delta = (base.loc[ix, 'valor_contabil'] * rng.uniform(.01, .05, 200)).astype(np.int64)
base.loc[ix, 'valor_contabil'] += delta
registra(base.loc[ix, 'id_bem'], 'E6', 'Valor contábil diferente de custo – depreciação', 'Precisão', delta.values)

gabarito = pd.DataFrame(gabarito)
print(gabarito.groupby('codigo_erro').size())

## Bloco 6 – Reajuste final
Os bens **sem erro** absorvem a diferença para que o custo e a depreciação continuem batendo com a nota 15. Os bens com erro ficam intactos. Também calcula a depreciação de 2025, ajustada ao valor da nota 15.

In [ ]:
# =====================================================================
# BLOCO 6 – Reajuste dos bens sem erro para fechar com a nota 15
# (os erros ficam intactos; só os bens corretos absorvem a diferença)
# =====================================================================
protegidos = set(base.loc[list(usados), 'id_bem'])   # inclui os originais das duplicatas (E1)
com_erro = base['id_bem'].isin(gabarito['id_bem']) | base['id_bem'].isin(protegidos)
for classe, (custo_mil, dep_mil, _) in NOTA15.items():
    m = base['classe'] == classe
    for col, alvo in (('custo_aquisicao', custo_mil * CENT), ('depreciacao_acumulada', dep_mil * CENT)):
        if alvo == 0 and col == 'depreciacao_acumulada':
            continue   # terrenos e obras: só os bens com erro (E3/E4) têm depreciação
        livre = m & ~com_erro
        falta = alvo - base.loc[m, col].sum()
        base.loc[livre, col] = ajusta_total(base.loc[livre, col].values.astype(float),
                                            int(base.loc[livre, col].sum() + falta))
    livre = m & ~com_erro
    base.loc[livre, 'depreciacao_acumulada'] = np.minimum(base.loc[livre, 'depreciacao_acumulada'], base.loc[livre, 'custo_aquisicao'])

nao_e6 = ~base['id_bem'].isin(gabarito.loc[gabarito['codigo_erro'] == 'E6', 'id_bem'])
base.loc[nao_e6, 'valor_contabil'] = base.loc[nao_e6, 'custo_aquisicao'] - base.loc[nao_e6, 'depreciacao_acumulada']

# Depreciação do exercício 2025 (cálculo linear simples, apenas informativo)
inicio = pd.to_datetime(base['data_inicio_depreciacao'])
meses = ((DATA_BASE - inicio.clip(lower=pd.Timestamp('2025-01-01'))).dt.days / 30.44).clip(0, 12)
dep25 = (base['custo_aquisicao'] / base['vida_util_anos'].replace(0, np.nan) * meses / 12).fillna(0)
base['depreciacao_2025'] = np.minimum(dep25, base['valor_contabil'].clip(lower=0)).round().astype(np.int64)
DEP2025 = {'Imóveis': 434_875, 'Máquinas, equipamentos e instalações': 4_123_031, 'Outros': 287_270}   # nota 15, R$ mil
for classe, alvo in DEP2025.items():
    m = (base['classe'] == classe) & (base['depreciacao_2025'] > 0)
    base.loc[m, 'depreciacao_2025'] = ajusta_total(base.loc[m, 'depreciacao_2025'].values.astype(float), alvo * CENT)
base['origem_dado'] = 'SINTÉTICO – trabalho acadêmico UnB – não representa registros da Suzano'
print('Base reajustada.')

## Bloco 7 – Conciliação e arquivos
Mostra a **conciliação** da base com a nota 15 e grava três arquivos: a base, os documentos suporte simulados (usados para testar a amostra) e o gabarito de erros.

In [ ]:
# =====================================================================
# BLOCO 7 – Conciliação com a nota 15 e gravação dos arquivos
# =====================================================================
for col in ('custo_aquisicao', 'depreciacao_acumulada', 'valor_contabil', 'depreciacao_2025'):
    base[col] = base[col] / 100          # centavos -> reais

conc = base.groupby('classe')[['custo_aquisicao', 'depreciacao_acumulada', 'valor_contabil']].sum() / 1000
conc['custo_nota15'] = [NOTA15[c][0] for c in conc.index]
conc['depr_nota15'] = [NOTA15[c][1] for c in conc.index]
conc['vc_nota15'] = conc['custo_nota15'] - conc['depr_nota15']
conc['dif_vc'] = (conc['valor_contabil'] - conc['vc_nota15']).round(2)
conc.loc['Total'] = conc.sum()
pd.set_option('display.float_format', '{:,.2f}'.format)
print('\nCONCILIAÇÃO (R$ mil)\n', conc[['custo_aquisicao', 'custo_nota15', 'depreciacao_acumulada', 'depr_nota15', 'valor_contabil', 'vc_nota15', 'dif_vc']])
print('\nDiferença de valor contábil explicada pelo erro E6 (R$ mil):', round(gabarito.loc[gabarito.codigo_erro == 'E6', 'distorcao_reais'].sum() / 1000, 2))

# Documento suporte simulado (valor da nota fiscal = custo correto), usado no teste da amostra
# valor do documento = custo final registrado, exceto nos bens com custo inflado (E10)
infl_e10 = gabarito.loc[gabarito.codigo_erro == 'E10'].set_index('id_bem')['distorcao_reais']
docs = pd.DataFrame({'id_bem': base['id_bem'], 'numero_nf': base['numero_nf'],
                     'data_documento': base['data_aquisicao'].dt.date,
                     'valor_documento': (base['custo_aquisicao'] - base['id_bem'].map(infl_e10).fillna(0)).round(2)})
docs.loc[base['id_bem'].isin(gabarito.loc[gabarito.codigo_erro == 'E9', 'id_bem']), 'valor_documento'] = np.nan   # sem documento
docs.loc[base['id_bem'].isin(gabarito.loc[gabarito.codigo_erro == 'E1', 'id_bem']), 'valor_documento'] = np.nan   # duplicata

colunas = ['id_bem', 'plaqueta', 'descricao', 'classe', 'subclasse', 'unidade', 'uf', 'centro_custo',
           'data_aquisicao', 'data_inicio_depreciacao', 'vida_util_anos', 'taxa_depreciacao_aa',
           'custo_aquisicao', 'depreciacao_acumulada', 'valor_contabil', 'depreciacao_2025',
           'fornecedor_id', 'numero_nf', 'data_lancamento', 'hora_lancamento', 'usuario_lancamento',
           'usuario_aprovador', 'em_garantia', 'situacao', 'origem_dado']
base = base[colunas]
base.to_csv('base_imobilizado_SIM.csv.gz', index=False, sep=';', decimal=',', encoding='utf-8-sig', compression='gzip')
docs.to_csv('documentos_suporte_SIM.csv.gz', index=False, sep=';', decimal=',', encoding='utf-8-sig', compression='gzip')
gabarito.to_csv('gabarito_erros_SIM.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
print('\nDistorção por tipo de erro (R$ mil):'); print((gabarito.groupby('codigo_erro')['distorcao_reais'].sum()/1000).round(1))
print(f'\nArquivos gravados. Linhas da base: {len(base):,} | erros no gabarito: {len(gabarito):,}')

## Bloco 8 – Download dos arquivos
Baixa os três arquivos para o seu computador (funciona apenas no Google Colab).

In [ ]:
from google.colab import files
for arq in ['base_imobilizado_SIM.csv.gz', 'documentos_suporte_SIM.csv.gz', 'gabarito_erros_SIM.csv']:
    files.download(arq)
